# Proyecto de interoperabilidad e integración de sistemas

## Caso: detección oportuna de estudiantes que requieren acompañamiento

| Campo | Valor |
|---|---|
| Grupo | Santiago Soto · Sergio Guzmán · Benis Gómez |
| Fecha de entrega | 23 de septiembre de 2026 |
| Fecha de sustentación | 24 de septiembre de 2026 |
| Entregable | este cuaderno (informe + prototipo ejecutable) |

### Pregunta central (sección 3 del enunciado)

¿Cómo puede la universidad utilizar información de sus diferentes sistemas para identificar oportunamente estudiantes que podrían requerir acompañamiento de Bienestar Universitario?

El equipo responde esa pregunta con un prototipo de integración. Académico, la plataforma virtual y Financiero ya registran hechos del estudiante. Hoy esos hechos no llegan a tiempo a Bienestar Institucional. La solución no reemplaza esos sistemas: los hace interoperar. Un correlador decide cuándo hay suficientes señales para abrir un caso de revisión.

### Índice

1. Instrucciones de ejecución del prototipo
2. Etapa 8. Análisis del problema
3. Etapa 9. Requisitos de integración
4. Etapa 10. Alternativas de solución
5. Etapa 11. Decisión arquitectónica
6. Etapa 12. Decisión tecnológica
7. Etapa 13. Diseño de la solución
8. Etapa 14. Prototipo funcional
9. Etapa 15. Escenario mínimo de demostración
10. Etapa 16. Pruebas
11. Limitaciones y evolución
12. Etapa 18. Preguntas de sustentación
13. Detención de clientes

# Instrucciones de ejecución

Este cuaderno es el informe del curso y, a la vez, el prototipo. No usar *Run All*. `input()` bloquea el kernel de Jupyter. Para la **demo de clase** ejecutar **Detector**, **Bienestar** y luego el **escenario automático** (sección 15). Las celdas con `input()` imitan el laboratorio de eventos; el escenario no las necesita.

Si se reejecutan las constantes, se detienen los clientes MQTT previos; no hacer Run All porque las celdas con `input()` esperan Enter.

## Dónde ejecutarlo

| Entorno | Cómo |
|---|---|
| VS Code / Cursor | abrir este `.ipynb`, kernel **Python 3**, ejecutar celdas en orden |
| Jupyter Notebook / Lab | mismo kernel; ejecutar desde la celda de `pip` en adelante para el prototipo |
| Google Colab | subir el archivo; aceptar la instalación de `paho-mqtt` |

## Dependencia

La primera celda de código del prototipo instala `paho-mqtt`. Matplotlib es opcional (diagrama de cajas). Si Matplotlib no está, el diagrama Mermaid y el recuadro ASCII bastan.

## Red

El prototipo **requiere Internet**. Publica y consume en `broker.emqx.io:1883`. Si el broker público no responde, el prototipo no corre. Ese riesgo se documenta en la decisión arquitectónica y en las preguntas de sustentación.

## Orden obligatorio

1. Instalar la librería y ejecutar constantes y funciones auxiliares.
2. Arrancar **Detector de riesgo** y **Bienestar Institucional** (`loop_start()` en hilos).
3. Publicar con el escenario automático, o con las celdas interactivas.
4. Ejecutar las cuatro pruebas de la etapa 16.
5. Detener hilos al final (`loop_stop` y `disconnect`).

Los consumidores **no** usan `loop_forever()` como única demostración, porque bloquearían el kernel y las pruebas no podrían seguir. En un kernel aparte, cada sistema sí podría usar `loop_forever()`, como en el laboratorio.

## Topics (espacio de nombres del grupo)

El broker es público. El prefijo `uni/soto-guzman-gomez/` evita colisiones con otros cursos.

```
uni/soto-guzman-gomez/academico/eventos
uni/soto-guzman-gomez/virtual/eventos
uni/soto-guzman-gomez/financiero/eventos
uni/soto-guzman-gomez/detector/alertas
uni/soto-guzman-gomez/bienestar/acciones
```

## Identificadores

Los estudiantes de prueba son ficticios (`EST001`, `EST002`, …). No se usan cédulas reales.

# 8. Análisis del problema

## 8.1 Problema institucional

### Qué ocurre actualmente

Bienestar Institucional debe acompañar a estudiantes en riesgo de desertar o de deteriorar su bienestar. La universidad ya genera indicios en tres sistemas que no conversan entre sí. El Sistema Académico registra la caída del promedio y las cancelaciones. La Plataforma Virtual registra inactividad y baja participación. El Sistema Financiero registra mora. Bienestar solo ve solicitudes, remisiones y atenciones propias.

En un mismo periodo, una estudiante puede bajar de 4.2 a 2.9, dejar de entrar al aula virtual dieciocho días y acumular mora. Cada hecho queda en su silo. Bienestar se entera cuando ella pide cita, cuando un docente remite o cuando el semestre ya avanzó.

### Por qué constituye un problema

No es un problema de falta de datos. Es un problema de **oportunidad** y de **visión parcial**. El dato existe, pero no cruza el límite del sistema que lo originó. Sin una vista suficiente, la universidad reacciona tarde.

### Quiénes se ven afectados

| Actor | Cómo le afecta |
|---|---|
| Estudiante | recibe apoyo cuando el daño académico o financiero ya es mayor |
| Bienestar Institucional | trabaja con remisiones tardías y sin contexto de otros sistemas |
| Docentes y programas | remiten casos aislados, sin saber si hay mora o inactividad virtual |
| La universidad | pierde oportunidad de permanencia con información que ya pagó por registrar |

### Consecuencias

La identificación tardía aumenta la probabilidad de cancelación masiva, pérdida de matrícula o deserción. El equipo de Bienestar satura su agenda con casos ya graves. Los sistemas siguen siendo “correctos” cada uno por separado: el fallo está en la integración.

### Situación que se espera mejorar

Bienestar debe enterarse **cuando la confluencia de hechos lo justifique**, sin reemplazar Académico, Virtual ni Financiero, y sin que un profesional tenga que consultar tres pantallas. El prototipo demuestra ese recorte: hechos al ocurrir, correlación mínima y una alerta accionable.

## 8.2 Sistemas y actores involucrados

El enunciado nombra cuatro sistemas. El equipo **los usa los cuatro**. El caso del estudiante no se reduce a una sola fuente: la oportunidad de Bienestar nace de combinar académico, virtual y financiero. Bienestar es el destinatario institucional, no un productor de esos hechos.

No es obligatorio, según el PDF, usar todos. Se usan los cuatro porque omitir uno dejaría ciega la regla de alerta. Un correlador de integración se añade más adelante; no es un quinto sistema de negocio de la universidad.

| Sistema | Tipo | Actor humano principal | ¿En el prototipo? | Justificación |
|---|---|---|---|---|
| Sistema Académico | sistema de información académica | registro / programa / docente | sí, productor | origina promedio, delta y cancelaciones |
| Plataforma Virtual | LMS / aula virtual | el estudiante al participar; soporte del LMS | sí, productor | origina inactividad y participación |
| Sistema Financiero | obligaciones y pagos | tesorería / cartera | sí, productor | origina mora y regularización |
| Bienestar Institucional | atención y acompañamiento | profesional de Bienestar | sí, consumidor | debe recibir la alerta, no los silos crudos |

El Detector de riesgo, descrito en las etapas 11 y 13, es un **componente nuevo de integración**. No sustituye a Bienestar ni califica al estudiante. Solo correlaciona señales y publica que un código ficticio requiere revisión.

## 8.3 Información involucrada

Se intercambia lo mínimo para decidir una revisión. No viajan cédulas, historias clínicas ni horarios completos.

| Información | Dónde se origina | Quién es responsable | Quién la necesita | Cuándo se necesita | Para qué |
|---|---|---|---|---|---|
| Promedio anterior y actual, y el delta | Sistema Académico | registro académico | el correlador; Bienestar solo ve el resumen en la alerta | cuando cierra un corte o se detecta la caída en el periodo | saber si hay `CAIDA_RENDIMIENTO` |
| Asignaturas canceladas, total y si es cancelación total | Sistema Académico | registro académico | el correlador | al confirmar la cancelación | señal `CANCELACION_ASIGNATURAS` o crítica |
| Días de inactividad y porcentaje de participación | Plataforma Virtual | administración del LMS | el correlador | al superarse el umbral de ausencia o de baja participación | señal `INACTIVIDAD_VIRTUAL` |
| Días de mora, valor pendiente y estado de la obligación | Sistema Financiero | cartera / tesorería | el correlador | al vencer el umbral de mora o al regularizar el pago | señal `MORA_FINANCIERA` o su retiro |
| Alerta de revisión (prioridad, señales, eventos origen) | correlador de integración | el componente Detector | Bienestar Institucional | en cuanto la regla se cumple o cambia | abrir, actualizar o cerrar el caso |
| Acuse `CasoRegistradoBienestar` | Bienestar Institucional | profesional / sistema de casos | trazabilidad de la demo | al recibir la alerta | evidenciar que el destino usó el mensaje |

El identificador canónico entre sistemas es `codigo_estudiante` ficticio (`EST001`, …). Sin ese identificador compartido no hay correlación.

## 8.4 Problemas de interoperabilidad

Estos problemas explican por qué Bienestar no ve a tiempo el caso del estudiante. No son defectos de un solo aplicativo.

| Problema | Qué implica en este caso |
|---|---|
| Silos | Académico, Virtual, Financiero y Bienestar operan de forma independiente. La caída de promedio no dispara nada en Bienestar. |
| Sistemas heterogéneos | El enunciado admite tecnologías y modelos de datos distintos. No se puede exigir un único esquema interno ni reescribir núcleos. |
| Tiempos distintos | Los hechos no ocurren juntos. La mora puede llegar días después de la inactividad. Una consulta puntual no basta. |
| Identificador | Si cada sistema usa una llave distinta, no se puede afirmar que es la misma estudiante. Hace falta un código canónico de intercambio. |
| Contrato | Hoy no hay un sobre común de “hecho ocurrido”. Cada consulta ad hoc inventa campos. El intercambio debe nombrar el hecho y su `data`. |
| Oportunidad | Un lote nocturno o una pantalla que el profesional abre “cuando puede” llega tarde para acompañar. El hecho debe poder avisarse al ocurrir. |
| Privacidad | Bienestar no necesita el detalle financiero completo ni el log del LMS. Debe llegar un resumen accionable, con el mínimo de datos personales. |

La solución de integración tiene que atacar esos siete puntos. Cualquier arquitectura que obligue a Bienestar a conocer las APIs internas de los tres silos, o que exija modificar sus núcleos, no resuelve el problema planteado por el PDF.

# 9. Requisitos de integración

Los requisitos salen del problema de la estudiante que no es vista a tiempo. **No salen de una tecnología previa.**

| ID | Requisito | Trazado al problema |
|---|---|---|
| REQ-01 | Los hechos relevantes deben poder intercambiarse **cuando ocurren**, no solo cuando alguien consulta. | La caída de promedio, la inactividad y la mora ya existen en silos; Bienestar no se entera. |
| REQ-02 | La alerta debe llegar con **oportunidad**: antes de un ciclo de consulta por lotes o de una remisión tardía. | El daño aumenta mientras Bienestar espera la solicitud del estudiante. |
| REQ-03 | No se modificarán los **núcleos** de Académico, Virtual, Financiero ni Bienestar. Solo se añaden adaptadores que publican o consumen. | La universidad no desea reemplazar sistemas heterogéneos. |
| REQ-04 | El intercambio usará un **contrato** estructurado (objeto con nombre de hecho, tiempo, origen y `data`). | Sin contrato, cada silo habla un idioma distinto. |
| REQ-05 | Un mensaje inválido se **rechaza y se registra**; el proceso no debe caer. | Datos incompletos no pueden tumbar la detección del resto de estudiantes. |
| REQ-06 | El **fallo de un consumidor** no detiene a los productores ni al correlador. | Si Bienestar está en mantenimiento, Académico debe seguir registrando hechos. |
| REQ-07 | Incorporar un sistema nuevo debe ser, en lo esencial, una **nueva suscripción** (o un nuevo productor), sin reescribir a los demás. | Pueden aparecer más fuentes (por ejemplo, biblioteca) sin rehacer Bienestar. |
| REQ-08 | Cada hecho lleva `event_id` para **trazabilidad**. La alerta cita los eventos origen. | Hay que explicar, en sustentación, de dónde salió el caso. |
| REQ-09 | Mínimo de datos personales: código ficticio, nombre de prueba y métricas del hecho. Nada de cédulas ni expedientes. | Privacidad: Bienestar no debe ver el silo crudo. |
| REQ-10 | Debe existir **correlación** por estudiante y periodo. Un hecho no crítico aislado no abre alerta; una señal crítica (cancelación total, inactividad ≥ 21 días o mora ≥ 30) también abre alerta. | El PDF pregunta cómo se decide que alguien requiere revisión. |
| REQ-11 | Un hecho de **actualización** (recuperación de promedio o pago regularizado) debe retirar la señal y cerrar o actualizar la alerta. | El estado del estudiante cambia; la alerta no puede quedar congelada. |

REQ-10 es el corazón del prototipo. Si una sola baja de nota abriera el caso, no habría interoperabilidad: habría un `if` local en Académico.

# 10. Alternativas de solución

## Alternativa A. Eventos asíncronos con correlador

### Cómo funcionaría

Cada sistema de origen emite un **hecho** cuando ocurre algo relevante para el periodo. El hecho viaja por un bus de integración. Un componente nuevo, el Detector de riesgo, escucha los hechos, mantiene estado en memoria por `codigo_estudiante` y aplica la regla de correlación. Si hay dos o más señales distintas, o una señal crítica, publica `EstudianteRequiereRevision`. Bienestar Institucional solo escucha esa alerta y registra el caso.

Los sistemas de origen no se llaman entre sí. Bienestar no consulta promedios ni mora. El correlador no es un sistema de negocio: no atiende al estudiante y no reemplaza a Bienestar.

### Componentes

- Productores: Sistema Académico, Plataforma Virtual, Sistema Financiero.
- Bus de hechos (canal de publicación y suscripción).
- Detector de riesgo (consumidor de los tres orígenes y productor de alertas).
- Bienestar Institucional (consumidor de alertas).

### Comunicación

Asíncrona y por contrato de mensaje. El productor no espera la respuesta de Bienestar. El consumidor procesa cuando el mensaje llega. Los tiempos distintos del PDF encajan: la mora puede publicarse días después de la inactividad y aun así entra al mismo estado.

### Ventajas

- Oportunidad: el hecho se avisa al ocurrir (REQ-01, REQ-02).
- Acoplamiento bajo: Académico no conoce a Bienestar (REQ-03, REQ-06).
- Extensión: un cuarto productor se suma al canal (REQ-07).
- Correlación natural en un solo componente (REQ-10, REQ-11).
- Privacidad: Bienestar no se suscribe a los topics crudos (REQ-09).

### Desventajas y limitaciones

- El estado del correlador, en este recorte, vive en memoria. Si el proceso muere, hay que reconstruir.
- La consistencia no es inmediata ni transaccional entre silos.
- Hace falta un contrato estable; un campo mal nombrado se rechaza.

### Riesgos

- Pérdida de mensaje si el destino no está escuchando y el canal no retiene.
- Correlación errónea si el identificador no es canónico.
- Umbrales mal calibrados (falsos positivos o ceguera).

### Adecuación al problema

El enunciado insiste en que no todo ocurre al mismo tiempo y que los sistemas no se reemplazan. Esta alternativa trata esos dos puntos de frente. La decisión de “requiere revisión” queda explícita en el correlador, que es justo lo que el PDF pide determinar.

## Alternativa B. Orquestación síncrona por consultas

### Cómo funcionaría

Bienestar, o un API Gateway delante de Bienestar, consultaría a Académico, a la Plataforma Virtual y a Financiero cuando un profesional (o un lote) pida “estudiantes en riesgo”. Cada sistema expondría un recurso. El orquestador esperaría las tres respuestas, aplicaría la regla y mostraría el caso.

### Componentes

- APIs en Académico, Virtual y Financiero.
- Orquestador o puerta de enlace.
- Cliente en Bienestar Institucional.

### Comunicación

Síncrona: petición y respuesta. Bienestar conoce direcciones, contratos y disponibilidad de los tres núcleos. El intercambio ocurre cuando alguien pregunta, no cuando el hecho ocurre.

### Ventajas

- Respuesta inmediata en la llamada exitosa.
- Contratos de consulta fáciles de documentar.
- El profesional puede forzar un refresco puntual del expediente.

### Desventajas y limitaciones

- Acoplamiento temporal: si Financiero no responde, el flujo se corta o queda incompleto.
- Bienestar queda acoplado a tres contratos internos (viola el espíritu de REQ-03).
- No hay aviso al ocurrir la caída de promedio: hay que preguntar (debilita REQ-01 y REQ-02).
- Incorporar un sistema nuevo implica cambiar el orquestador (debilita REQ-07).
- Para imitar oportunidad haría falta un sondeo continuo, costoso y frágil.

### Riesgos

- Cascada de fallos.
- Timeouts que dejan a Bienestar con visión parcial, que es exactamente el problema actual.
- Presión para “abrir un poco” los núcleos y terminar modificándolos.

### Adecuación al problema

Resuelve la consulta a demanda. No resuelve la identificación tardía. El PDF describe hechos que aparecen en momentos distintos. Una arquitectura que solo pregunta cuando Bienestar tiene tiempo reproduce el retraso institucional.

## Comparación y selección

| Criterio | Alternativa A (eventos + correlador) | Alternativa B (consultas síncronas) |
|---|---|---|
| Oportunidad | el hecho se emite al ocurrir | la información viaja cuando alguien consulta |
| Acoplamiento | productores no conocen a Bienestar | Bienestar (o el gateway) conoce tres APIs |
| Fallo de un sistema | el resto sigue publicando o escuchando | una API caída corta o degrada el expediente |
| Incorporación de un sistema nuevo | nueva publicación o suscripción | cambiar el orquestador y sus llamadas |
| Adecuación al PDF | tiempos distintos y núcleos intocados | útil para consulta, débil para detección oportuna |

**Selección: alternativa A.**

La coherencia pedida por el enunciado es PROBLEMA → REQUISITOS → ARQUITECTURA. El problema es que Bienestar ve tarde una confluencia de hechos ya registrados. Los requisitos de oportunidad, de no tocar núcleos, de no detener productores y de correlacionar (REQ-01, REQ-02, REQ-03, REQ-06, REQ-07, REQ-10) se cumplen con eventos asíncronos y un correlador. La alternativa B optimizaría una pantalla de consulta; no el momento en que la estudiante empieza a desaparecer del aula virtual.

La tecnología concreta se elige **después**, en la etapa 12. Aquí solo se fija el estilo de integración.

# 11. Decisión arquitectónica

Se implementa la alternativa A. El recorte es representativo: tres productores, un correlador y un consumidor institucional.

### Responsabilidades

| Componente | Responsabilidad |
|---|---|
| Sistema Académico | emitir hechos de rendimiento y cancelación del periodo |
| Plataforma Virtual | emitir hechos de inactividad o baja participación |
| Sistema Financiero | emitir mora y regularización de pago |
| Detector de riesgo | validar, correlacionar en memoria y emitir alerta, actualización o cierre |
| Bienestar Institucional | recibir la alerta y registrar el caso de revisión |
| Canal de integración | transportar mensajes sin que el origen conozca al destino |

### Comunicación asíncrona

El productor publica y sigue. El Detector reacciona en su propio tiempo. Bienestar reacciona en el suyo. Esa independencia de tiempo cubre el “no todo ocurre al mismo tiempo” del enunciado.

### Independencia

Académico no invoca a Bienestar. Virtual no sabe si existe mora. El Detector no llama APIs de los núcleos: solo escucha. Si Bienestar se detiene, los hechos siguen emitiéndose.

### Datos

Sobre común: `event_id`, `event_name`, `timestamp`, `source_system`, `periodo`, `data`. Identificador canónico: `codigo_estudiante`. Bienestar recibe señales ya resumidas, no el detalle de cartera ni el log del LMS.

### Fallos

Un payload inválido se rechaza con registro y no tumba el Detector (REQ-05). Un consumidor caído no frena a los productores (REQ-06). El riesgo restante es la pérdida de mensajes si el destino no está presente y el canal no retiene. Se asume y se muestra en pruebas.

### Crecimiento

Un sistema nuevo publica en su propio canal o se suscribe al de alertas. El correlador puede añadir una señal sin reescribir a Bienestar. El límite actual es el estado en memoria y un solo proceso de Detector.

### Diagrama de arquitectura

```mermaid
flowchart LR
    A[Sistema Académico] --> C[Canal de hechos]
    V[Plataforma Virtual] --> C
    F[Sistema Financiero] --> C
    C --> D[Detector de riesgo]
    D --> L[Canal de alertas]
    L --> B[Bienestar Institucional]
```

```
  Académico ----hechos---+
  Virtual   ----hechos---+---> [Canal de integración] ---> Detector
  Financiero----hechos---+              |                    |
                                        |                    v
                                        +<--- alertas -------+
                                        |
                                        v
                               Bienestar Institucional
```

Los nombres de producto y de protocolo aparecen en la etapa 12, no aquí.

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

    fig, ax = plt.subplots(figsize=(11, 5.2))
    ax.set_xlim(0, 11)
    ax.set_ylim(0, 5.4)
    ax.axis("off")
    ax.set_title("Arquitectura: hechos asíncronos y correlador (caso Bienestar)")

    def caja(x, y, w, h, texto, color):
        parche = FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.08,rounding_size=0.12",
            facecolor=color, edgecolor="#222222", linewidth=1.2,
        )
        ax.add_patch(parche)
        ax.text(x + w / 2, y + h / 2, texto, ha="center", va="center", fontsize=9)

    caja(0.3, 3.8, 2.4, 1.0, "Sistema\nAcadémico", "#d9edf7")
    caja(0.3, 2.2, 2.4, 1.0, "Plataforma\nVirtual", "#d9edf7")
    caja(0.3, 0.6, 2.4, 1.0, "Sistema\nFinanciero", "#d9edf7")
    caja(4.0, 1.8, 2.8, 1.6, "Canal de\nintegración", "#fcf8e3")
    caja(7.6, 3.2, 2.8, 1.3, "Detector\nde riesgo", "#dff0d8")
    caja(7.6, 0.7, 2.8, 1.3, "Bienestar\nInstitucional", "#f2dede")

    estilo = dict(arrowstyle="->", mutation_scale=12, linewidth=1.2, color="#333333")
    ax.add_patch(FancyArrowPatch((2.7, 4.3), (4.0, 3.1), **estilo))
    ax.add_patch(FancyArrowPatch((2.7, 2.7), (4.0, 2.7), **estilo))
    ax.add_patch(FancyArrowPatch((2.7, 1.1), (4.0, 2.2), **estilo))
    ax.add_patch(FancyArrowPatch((6.8, 3.0), (7.6, 3.7), **estilo))
    ax.add_patch(FancyArrowPatch((9.0, 3.2), (9.0, 2.0), **estilo))

    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("AVISO: no se pudo generar el diagrama matplotlib.")
    print("El diagrama Mermaid y el recuadro ASCII de la etapa 11 bastan.")
    print(exc)

# 12. Decisión tecnológica

La arquitectura ya está fijada (eventos asíncronos y correlador). Ahora se eligen herramientas **solo para el prototipo de curso**.

### Qué se necesitaba

| Función en la alternativa A | Qué se evalúa |
|---|---|
| Canal de publicación y suscripción | MQTT, REST con colas, o una plataforma de registro (Kafka) |
| Cliente en Python | `paho-mqtt` u otro cliente |
| Contrato del mensaje | JSON o XML |
| Dónde corre el canal | broker público EMQX o Mosquitto local |
| Lenguaje del prototipo | Python en este cuaderno |

### Criterios

1. Fidelidad a la alternativa A (publicar / suscribir, asíncrono, varios consumidores).
2. Esfuerzo de prototipo (unas celdas, no un clúster).
3. Visibilidad del mensaje en la demostración (banners y JSON impreso).
4. Cero infraestructura institucional que el equipo no controla.

### Comparación breve

| Opción | Fidelidad a A | Esfuerzo | Visibilidad | Infraestructura |
|---|---|---|---|---|
| MQTT + EMQX público + `paho-mqtt` | alta | bajo | alta (payload impreso) | ninguna local |
| REST síncrono | baja: vuelve a la alternativa B | bajo | media | un servidor por sistema |
| Kafka | alta | alto para un curso | media | brokers, topics, runtime |
| Mosquitto en el portátil | alta | medio (instalar y exponer) | alta | depende de cada máquina |
| XML | equivalente | peor para la demo | peor lectura | — |

**Selección:** broker público `broker.emqx.io:1883`, cliente `paho-mqtt` (`CallbackAPIVersion.VERSION2`), QoS 1, payload JSON UTF-8, Python 3 en Jupyter.

`paho-mqtt` es suficiente: conexión, `publish` con `wait_for_publish()`, callbacks y `loop_start()`. JSON es el contrato legible en sustentación y coincide con el laboratorio de eventos del grupo. El broker público cumple el criterio de cero infraestructura.

**Limitación explícita:** este MQTT **no** es el bus institucional de la universidad. Es un canal de demostración. En producción haría falta un broker propio, autenticación y políticas de retención. Si EMQX público falla, el prototipo no corre; por eso la demo depende de Internet.

# 13. Diseño de la solución

## Flujo de información

El recorrido que pide el enunciado: dónde se genera el hecho → cómo se intercambia → cómo se procesa → quién lo recibe → qué ocurre después.

```mermaid
sequenceDiagram
    participant A as Sistema Académico
    participant V as Plataforma Virtual
    participant F as Sistema Financiero
    participant D as Detector de riesgo
    participant W as Bienestar Institucional

    A->>D: publish RendimientoAcademicoCaido
    Note over D: valida, 1 señal, no alerta
    V->>D: publish InactividadPlataformaDetectada
    Note over D: valida, correlaciona, 2 señales
    D->>W: publish EstudianteRequiereRevision
    W->>W: imprime caso y JSON CasoRegistradoBienestar
    F->>D: publish MoraFinancieraDetectada
    D->>W: publish AlertaActualizada
```

1. El hecho ocurre en el sistema de origen (corte académico, inactividad, mora).
2. El adaptador publica el sobre JSON en su topic.
3. El Detector valida. Si el mensaje es inválido, imprime `EVENTO_RECHAZADO` y sigue.
4. Si es válido, actualiza `ESTADO[codigo]` y evalúa señales.
5. Con dos señales o una crítica, publica la alerta. Bienestar imprime el caso y un acuse JSON.

## Componentes

| Componente | Responsabilidad | Recibe | Produce | Relaciones |
|---|---|---|---|---|
| Sistema Académico | emitir hechos académicos del periodo | entrada de demo (teclado o función) | `RendimientoAcademicoCaido`, `AsignaturasCanceladas`, `RendimientoRecuperado` | no conoce a Virtual, Financiero ni Bienestar |
| Plataforma Virtual | emitir inactividad o baja participación | entrada de demo | `InactividadPlataformaDetectada` | no conoce a Académico ni a Financiero |
| Sistema Financiero | emitir mora y regularización | entrada de demo | `MoraFinancieraDetectada`, `PagoRegularizado` | no conoce a Académico ni a Virtual |
| Detector de riesgo | validar, correlacionar, decidir alerta | los tres topics de eventos | `EstudianteRequiereRevision`, `AlertaActualizada`, `AlertaCerrada` | escucha orígenes; no llama APIs |
| Bienestar Institucional | registrar el caso de revisión | topic de alertas | JSON `CasoRegistradoBienestar` (acuse) | no se suscribe a eventos crudos |

## Contratos de información

Sobre común de todos los eventos:

```json
{
  "event_id": "uuid",
  "event_name": "NombreDelHecho",
  "timestamp": "2026-09-20T15:00:00",
  "source_system": "academico",
  "periodo": "2026-2",
  "data": {}
}
```

Validación en el Detector: JSON parseable; existe `event_name`; `data.codigo_estudiante` no vacío; campos numéricos de `data` con valor ≥ 0 (el `delta_promedio` sí puede ser negativo). Inválido → `EVENTO_RECHAZADO` y el proceso continúa.

### Académico — RendimientoAcademicoCaido

```json
{
  "event_id": "uuid",
  "event_name": "RendimientoAcademicoCaido",
  "timestamp": "2026-09-20T15:00:00",
  "source_system": "academico",
  "periodo": "2026-2",
  "data": {
    "codigo_estudiante": "EST001",
    "nombre": "Ana Pérez",
    "programa": "Ingeniería de Sistemas",
    "promedio_anterior": 4.2,
    "promedio_actual": 2.9,
    "delta_promedio": -1.3
  }
}
```

### Académico — AsignaturasCanceladas

```json
{
  "event_id": "uuid",
  "event_name": "AsignaturasCanceladas",
  "timestamp": "2026-09-20T15:01:00",
  "source_system": "academico",
  "periodo": "2026-2",
  "data": {
    "codigo_estudiante": "EST001",
    "nombre": "Ana Pérez",
    "asignaturas_canceladas": ["Cálculo I", "Física"],
    "total_canceladas": 2,
    "cancelacion_total": false
  }
}
```

### Académico — RendimientoRecuperado

```json
{
  "event_id": "uuid",
  "event_name": "RendimientoRecuperado",
  "timestamp": "2026-09-20T16:00:00",
  "source_system": "academico",
  "periodo": "2026-2",
  "data": {
    "codigo_estudiante": "EST001",
    "nombre": "Ana Pérez",
    "promedio_actual": 3.8
  }
}
```

### Virtual — InactividadPlataformaDetectada

```json
{
  "event_id": "uuid",
  "event_name": "InactividadPlataformaDetectada",
  "timestamp": "2026-09-20T15:02:00",
  "source_system": "virtual",
  "periodo": "2026-2",
  "data": {
    "codigo_estudiante": "EST001",
    "nombre": "Ana Pérez",
    "dias_inactividad": 18,
    "participacion_pct": 22
  }
}
```

### Financiero — MoraFinancieraDetectada

```json
{
  "event_id": "uuid",
  "event_name": "MoraFinancieraDetectada",
  "timestamp": "2026-09-20T15:03:00",
  "source_system": "financiero",
  "periodo": "2026-2",
  "data": {
    "codigo_estudiante": "EST001",
    "nombre": "Ana Pérez",
    "dias_mora": 20,
    "valor_pendiente": 850000,
    "estado_obligacion": "EN_MORA"
  }
}
```

### Financiero — PagoRegularizado

```json
{
  "event_id": "uuid",
  "event_name": "PagoRegularizado",
  "timestamp": "2026-09-20T16:10:00",
  "source_system": "financiero",
  "periodo": "2026-2",
  "data": {
    "codigo_estudiante": "EST001",
    "nombre": "Ana Pérez",
    "dias_mora": 0,
    "valor_pendiente": 0,
    "estado_obligacion": "AL_DIA"
  }
}
```

### Detector — EstudianteRequiereRevision

```json
{
  "event_id": "uuid",
  "event_name": "EstudianteRequiereRevision",
  "timestamp": "2026-09-20T15:02:05",
  "source_system": "detector",
  "periodo": "2026-2",
  "data": {
    "codigo_estudiante": "EST001",
    "nombre": "Ana Pérez",
    "nivel_prioridad": "MEDIA",
    "motivo": "Confluencia de señales en el mismo periodo",
    "senales": ["CAIDA_RENDIMIENTO", "INACTIVIDAD_VIRTUAL"],
    "eventos_origen": ["uuid-acad", "uuid-virt"],
    "accion_sugerida": "Revisión por profesional de Bienestar"
  }
}
```

### Detector — AlertaActualizada

```json
{
  "event_id": "uuid",
  "event_name": "AlertaActualizada",
  "timestamp": "2026-09-20T15:03:05",
  "source_system": "detector",
  "periodo": "2026-2",
  "data": {
    "codigo_estudiante": "EST001",
    "nombre": "Ana Pérez",
    "nivel_prioridad": "ALTA",
    "motivo": "Cambio en las señales del periodo",
    "senales": ["CAIDA_RENDIMIENTO", "INACTIVIDAD_VIRTUAL", "MORA_FINANCIERA"],
    "eventos_origen": ["uuid-acad", "uuid-virt", "uuid-fin"],
    "accion_sugerida": "Revisión por profesional de Bienestar"
  }
}
```

### Detector — AlertaCerrada

```json
{
  "event_id": "uuid",
  "event_name": "AlertaCerrada",
  "timestamp": "2026-09-20T16:00:05",
  "source_system": "detector",
  "periodo": "2026-2",
  "data": {
    "codigo_estudiante": "EST001",
    "nombre": "Ana Pérez",
    "nivel_prioridad": "MEDIA",
    "motivo": "Las señales vigentes ya no cumplen la regla de alerta",
    "senales": ["INACTIVIDAD_VIRTUAL"],
    "eventos_origen": ["uuid-acad", "uuid-virt", "uuid-rec"],
    "accion_sugerida": "Cerrar o reevaluar el caso en Bienestar"
  }
}
```

### Bienestar — acuse

```json
{
  "action": "CasoRegistradoBienestar",
  "event_id_origen": "uuid-alerta",
  "codigo_estudiante": "EST001",
  "estado_caso": "PENDIENTE_REVISION",
  "timestamp_procesamiento": "2026-09-20T15:02:06"
}
```

# 14. Prototipo funcional

**Alcance.** Stubs de Académico, Virtual y Financiero que publican hechos reales por MQTT. Detector y Bienestar son procesos independientes en el mismo kernel, con hilos (`loop_start()`). No hay GUI. No hay base de datos. No se reimplementan los sistemas universitarios.

**Qué demuestra.** Interoperabilidad real: un `publish` en un topic es recibido por otro cliente. La regla de alerta es explícita. Un solo evento no abre caso. Dos señales sí. Una crítica también.

**Qué no demuestra.** Un broker institucional, persistencia, autenticación ni identidad federada.

**Demo de clase.** Ejecutar la instalación, las constantes, las funciones, el Detector, Bienestar y el escenario de la sección 15. No ejecutar las celdas de `input()` en esa demo; quedan para el análogo del laboratorio.

In [ ]:
!pip -q install "paho-mqtt>=2.0"

In [ ]:
import json
import time
import uuid
from datetime import datetime

import paho.mqtt.client as mqtt

BROKER = "broker.emqx.io"
PORT = 1883
PERIODO = "2026-2"

TOPICS = {
    "academico": "uni/soto-guzman-gomez/academico/eventos",
    "virtual": "uni/soto-guzman-gomez/virtual/eventos",
    "financiero": "uni/soto-guzman-gomez/financiero/eventos",
    "detector": "uni/soto-guzman-gomez/detector/alertas",
    "bienestar": "uni/soto-guzman-gomez/bienestar/acciones",
}

UMBRAL_DELTA = -1.0
UMBRAL_CANCELADAS = 2
UMBRAL_INACTIVIDAD = 14
UMBRAL_PARTICIPACION = 30
UMBRAL_MORA = 15
CRITICA_INACTIVIDAD = 21
CRITICA_MORA = 30

CAMPOS_NUMERICOS_NO_NEGATIVOS = (
    "promedio_anterior",
    "promedio_actual",
    "total_canceladas",
    "dias_inactividad",
    "participacion_pct",
    "dias_mora",
    "valor_pendiente",
)

# Si se reejecuta esta celda, detener clientes MQTT previos (loop_stop+disconnect) para no dejar hilos huérfanos.
if "CLIENTES" in globals() and CLIENTES:
    for _nombre, _cliente in list(CLIENTES.items()):
        try:
            _cliente.loop_stop()
        except Exception:
            pass
        try:
            _cliente.disconnect()
        except Exception:
            pass
        print(f"Cliente previo '{_nombre}' detenido.")

ESTADO = {}
CLIENTES = {}

print("Constantes listas. Broker:", BROKER, PORT)
print("Topics:")
for nombre, topic in TOPICS.items():
    print(f"  {nombre:12s} {topic}")

In [ ]:
def reset_estado():
    ESTADO.clear()
    print("ESTADO reiniciado.")


def envoltorio(event_name, source_system, periodo, data):
    return {
        "event_id": str(uuid.uuid4()),
        "event_name": event_name,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "source_system": source_system,
        "periodo": periodo,
        "data": data,
    }


def validar(evento):
    if not isinstance(evento, dict):
        return False, "el payload no es un objeto JSON"
    if not evento.get("event_name"):
        return False, "falta event_name"
    data = evento.get("data")
    if not isinstance(data, dict):
        return False, "falta data u objeto data inválido"
    codigo = data.get("codigo_estudiante")
    if codigo is None or not str(codigo).strip():
        return False, "data.codigo_estudiante vacío o ausente"
    for campo in CAMPOS_NUMERICOS_NO_NEGATIVOS:
        if campo not in data:
            continue
        valor = data[campo]
        if isinstance(valor, bool) or not isinstance(valor, (int, float)):
            return False, f"{campo} no es numérico"
        if valor < 0:
            return False, f"{campo} es negativo"
    if "delta_promedio" in data:
        valor = data["delta_promedio"]
        if isinstance(valor, bool) or not isinstance(valor, (int, float)):
            return False, "delta_promedio no es numérico"
    return True, "ok"


def evaluar_senales(evento):
    nombre = evento.get("event_name")
    data = evento.get("data") or {}
    agregar = set()
    quitar = set()
    criticas_add = set()
    criticas_del = set()

    if nombre == "RendimientoAcademicoCaido":
        delta = data.get("delta_promedio")
        if isinstance(delta, (int, float)) and not isinstance(delta, bool) and delta <= UMBRAL_DELTA:
            agregar.add("CAIDA_RENDIMIENTO")
    elif nombre == "AsignaturasCanceladas":
        total = data.get("total_canceladas", 0)
        if isinstance(total, (int, float)) and total >= UMBRAL_CANCELADAS:
            agregar.add("CANCELACION_ASIGNATURAS")
        if data.get("cancelacion_total") is True:
            agregar.add("CANCELACION_ASIGNATURAS")
            criticas_add.add("CANCELACION_TOTAL")
    elif nombre == "RendimientoRecuperado":
        quitar.add("CAIDA_RENDIMIENTO")
    elif nombre == "InactividadPlataformaDetectada":
        dias = data.get("dias_inactividad", 0) or 0
        participacion = data.get("participacion_pct", 100)
        if participacion is None:
            participacion = 100
        if dias >= UMBRAL_INACTIVIDAD or participacion < UMBRAL_PARTICIPACION:
            agregar.add("INACTIVIDAD_VIRTUAL")
        if dias >= CRITICA_INACTIVIDAD:
            criticas_add.add("INACTIVIDAD_PROLONGADA")
    elif nombre == "MoraFinancieraDetectada":
        dias = data.get("dias_mora", 0) or 0
        if dias >= UMBRAL_MORA:
            agregar.add("MORA_FINANCIERA")
        if dias >= CRITICA_MORA:
            criticas_add.add("MORA_GRAVE")
    elif nombre == "PagoRegularizado":
        quitar.add("MORA_FINANCIERA")
        criticas_del.add("MORA_GRAVE")

    return agregar, quitar, criticas_add, criticas_del


def detener_cliente(nombre):
    cliente = CLIENTES.pop(nombre, None)
    if cliente is None:
        return
    try:
        cliente.loop_stop()
    except Exception:
        pass
    try:
        cliente.disconnect()
    except Exception:
        pass
    print(f"Cliente '{nombre}' detenido.")


def detener_todos():
    for nombre in list(CLIENTES):
        detener_cliente(nombre)


def publicar(topic, evento):
    cliente = mqtt.Client(
        mqtt.CallbackAPIVersion.VERSION2,
        client_id=f"pub_{uuid.uuid4().hex[:8]}",
    )
    cliente.connect(BROKER, PORT, 60)
    cliente.loop_start()
    mensaje = json.dumps(evento, indent=4, ensure_ascii=False)
    resultado = cliente.publish(topic, mensaje, qos=1)
    try:
        resultado.wait_for_publish(timeout=5)
    except Exception:
        print("Publicación no confirmada en 5 s (broker lento o inalcanzable). No se espera más.")
    print(mensaje)
    cliente.loop_stop()
    cliente.disconnect()
    return evento


def publicar_crudo(topic, payload):
    cliente = mqtt.Client(
        mqtt.CallbackAPIVersion.VERSION2,
        client_id=f"pub_{uuid.uuid4().hex[:8]}",
    )
    cliente.connect(BROKER, PORT, 60)
    cliente.loop_start()
    resultado = cliente.publish(topic, payload, qos=1)
    try:
        resultado.wait_for_publish(timeout=5)
    except Exception:
        print("Publicación no confirmada en 5 s (broker lento o inalcanzable). No se espera más.")
    print(payload if isinstance(payload, str) else repr(payload))
    cliente.loop_stop()
    cliente.disconnect()


def publicar_rendimiento(codigo, nombre, promedio_anterior, promedio_actual,
                         programa="Ingeniería de Sistemas", periodo=PERIODO):
    anterior = float(promedio_anterior)
    actual = float(promedio_actual)
    delta = round(actual - anterior, 2)
    evento = envoltorio(
        "RendimientoAcademicoCaido",
        "academico",
        periodo,
        {
            "codigo_estudiante": codigo,
            "nombre": nombre,
            "programa": programa,
            "promedio_anterior": anterior,
            "promedio_actual": actual,
            "delta_promedio": delta,
        },
    )
    print("==== SISTEMA ACADÉMICO ====")
    print("Hecho: RendimientoAcademicoCaido")
    return publicar(TOPICS["academico"], evento)


def publicar_cancelaciones(codigo, nombre, asignaturas, total_canceladas,
                           cancelacion_total=False, periodo=PERIODO):
    evento = envoltorio(
        "AsignaturasCanceladas",
        "academico",
        periodo,
        {
            "codigo_estudiante": codigo,
            "nombre": nombre,
            "asignaturas_canceladas": list(asignaturas),
            "total_canceladas": int(total_canceladas),
            "cancelacion_total": bool(cancelacion_total),
        },
    )
    print("==== SISTEMA ACADÉMICO ====")
    print("Hecho: AsignaturasCanceladas")
    return publicar(TOPICS["academico"], evento)


def publicar_recuperado(codigo, nombre, promedio_actual, periodo=PERIODO):
    evento = envoltorio(
        "RendimientoRecuperado",
        "academico",
        periodo,
        {
            "codigo_estudiante": codigo,
            "nombre": nombre,
            "promedio_actual": float(promedio_actual),
        },
    )
    print("==== SISTEMA ACADÉMICO ====")
    print("Hecho: RendimientoRecuperado")
    return publicar(TOPICS["academico"], evento)


def publicar_inactividad(codigo, nombre, dias_inactividad, participacion_pct=50, periodo=PERIODO):
    evento = envoltorio(
        "InactividadPlataformaDetectada",
        "virtual",
        periodo,
        {
            "codigo_estudiante": codigo,
            "nombre": nombre,
            "dias_inactividad": int(dias_inactividad),
            "participacion_pct": float(participacion_pct),
        },
    )
    print("==== PLATAFORMA VIRTUAL ====")
    print("Hecho: InactividadPlataformaDetectada")
    return publicar(TOPICS["virtual"], evento)


def publicar_mora(codigo, nombre, dias_mora, valor_pendiente=0, periodo=PERIODO):
    evento = envoltorio(
        "MoraFinancieraDetectada",
        "financiero",
        periodo,
        {
            "codigo_estudiante": codigo,
            "nombre": nombre,
            "dias_mora": int(dias_mora),
            "valor_pendiente": float(valor_pendiente),
            "estado_obligacion": "EN_MORA",
        },
    )
    print("==== SISTEMA FINANCIERO ====")
    print("Hecho: MoraFinancieraDetectada")
    return publicar(TOPICS["financiero"], evento)


def publicar_pago_regularizado(codigo, nombre, periodo=PERIODO):
    evento = envoltorio(
        "PagoRegularizado",
        "financiero",
        periodo,
        {
            "codigo_estudiante": codigo,
            "nombre": nombre,
            "dias_mora": 0,
            "valor_pendiente": 0,
            "estado_obligacion": "AL_DIA",
        },
    )
    print("==== SISTEMA FINANCIERO ====")
    print("Hecho: PagoRegularizado")
    return publicar(TOPICS["financiero"], evento)


print("Funciones auxiliares listas.")

## Detector de riesgo

Consume los tres topics de eventos. Valida. Actualiza `ESTADO` en memoria. Publica alertas en `uni/soto-guzman-gomez/detector/alertas`.

Usa `loop_start()` para no bloquear el kernel. En un kernel aparte, el equivalente de laboratorio sería `loop_forever()` después de `connect`.

In [ ]:
def _publicar_alerta(event_name, codigo, estado, prioridad, motivo):
    evento = envoltorio(
        event_name,
        "detector",
        estado.get("periodo", PERIODO),
        {
            "codigo_estudiante": codigo,
            "nombre": estado.get("nombre", ""),
            "nivel_prioridad": prioridad,
            "motivo": motivo,
            "senales": sorted(estado.get("senales", set())),
            "eventos_origen": list(estado.get("eventos", [])),
            "accion_sugerida": (
                "Cerrar o reevaluar el caso en Bienestar"
                if event_name == "AlertaCerrada"
                else "Revisión por profesional de Bienestar"
            ),
        },
    )
    print("==== DETECTOR DE RIESGO ====")
    print("Alerta:", event_name, "| estudiante:", codigo, "| prioridad:", prioridad)
    publicar(TOPICS["detector"], evento)
    return evento


def procesar_evento(evento):
    ok, razon = validar(evento)
    if not ok:
        print("EVENTO_RECHAZADO:", razon)
        try:
            print(json.dumps(evento, indent=4, ensure_ascii=False))
        except Exception:
            print(evento)
        return

    data = evento["data"]
    codigo = str(data["codigo_estudiante"]).strip()
    agregar, quitar, criticas_add, criticas_del = evaluar_senales(evento)

    estado = ESTADO.setdefault(
        codigo,
        {
            "senales": set(),
            "criticas": set(),
            "eventos": [],
            "alerta_activa": False,
            "nombre": "",
            "periodo": evento.get("periodo", PERIODO),
        },
    )
    if data.get("nombre"):
        estado["nombre"] = data["nombre"]
    if evento.get("periodo"):
        estado["periodo"] = evento["periodo"]

    senales_antes = set(estado["senales"])
    criticas_antes = set(estado["criticas"])
    n_antes = len(senales_antes)
    hay_critica_antes = bool(criticas_antes)
    prioridad_antes = "ALTA" if (hay_critica_antes or n_antes >= 3) else "MEDIA"

    estado["senales"] |= agregar
    estado["senales"] -= quitar
    estado["criticas"] |= criticas_add
    estado["criticas"] -= criticas_del
    if evento.get("event_id"):
        estado["eventos"].append(evento["event_id"])

    n_senales = len(estado["senales"])
    hay_critica = bool(estado["criticas"])
    cumple = n_senales >= 2 or hay_critica
    prioridad = "ALTA" if (hay_critica or n_senales >= 3) else "MEDIA"

    print("==== DETECTOR DE RIESGO ====")
    print("Estudiante:", codigo, "| señales:", sorted(estado["senales"]), "| críticas:", sorted(estado["criticas"]))

    if not cumple:
        if estado["alerta_activa"]:
            _publicar_alerta(
                "AlertaCerrada",
                codigo,
                estado,
                "MEDIA",
                "Las señales vigentes ya no cumplen la regla de alerta",
            )
            estado["alerta_activa"] = False
        else:
            print("Un solo conjunto insuficiente. No hay alerta (hace falta correlación o una crítica).")
        return

    if not estado["alerta_activa"]:
        _publicar_alerta(
            "EstudianteRequiereRevision",
            codigo,
            estado,
            prioridad,
            "Confluencia de señales en el mismo periodo",
        )
        estado["alerta_activa"] = True
    else:
        senales_cambiaron = estado["senales"] != senales_antes or estado["criticas"] != criticas_antes
        if senales_cambiaron or prioridad != prioridad_antes:
            _publicar_alerta(
                "AlertaActualizada",
                codigo,
                estado,
                prioridad,
                "Cambio en las señales del periodo",
            )
        else:
            print("Señales y prioridad sin cambio. No se republica AlertaActualizada.")


def iniciar_detector():
    detener_cliente("detector")

    def al_conectar(client, userdata, flags, reason_code, properties):
        fallo = getattr(reason_code, "is_failure", None)
        if fallo is True or (fallo is None and reason_code != 0):
            print("DETECTOR DE RIESGO: conexión fallida, no se suscribe:", reason_code)
            return
        print("DETECTOR DE RIESGO conectado al broker MQTT")
        client.subscribe(TOPICS["academico"], qos=1)
        client.subscribe(TOPICS["virtual"], qos=1)
        client.subscribe(TOPICS["financiero"], qos=1)
        print("Suscrito a Académico, Virtual y Financiero.")

    def al_recibir(client, userdata, msg):
        try:
            texto = msg.payload.decode("utf-8")
            evento = json.loads(texto)
        except Exception as exc:
            print("EVENTO_RECHAZADO: JSON no parseable:", exc)
            return
        try:
            procesar_evento(evento)
        except Exception as exc:
            print("EVENTO_RECHAZADO: error de procesamiento:", exc)

    cliente = mqtt.Client(
        mqtt.CallbackAPIVersion.VERSION2,
        client_id=f"detector_{uuid.uuid4().hex[:8]}",
    )
    cliente.on_connect = al_conectar
    cliente.on_message = al_recibir
    cliente.connect(BROKER, PORT, 60)
    cliente.loop_start()
    CLIENTES["detector"] = cliente
    time.sleep(0.8)
    print("================================")
    print("     DETECTOR DE RIESGO")
    print("================================")
    print("Escuchando con loop_start(). El kernel sigue libre.")
    print("Nota: en un kernel aparte se usaría loop_forever(), como en el laboratorio.")
    return cliente


iniciar_detector()

## Bienestar Institucional

Consume solo el topic de alertas. Imprime el caso y un JSON de acuse `CasoRegistradoBienestar`. Publica el acuse en `uni/soto-guzman-gomez/bienestar/acciones`. No ve eventos crudos de los silos.

In [ ]:
def iniciar_bienestar():
    detener_cliente("bienestar")

    def al_conectar(client, userdata, flags, reason_code, properties):
        fallo = getattr(reason_code, "is_failure", None)
        if fallo is True or (fallo is None and reason_code != 0):
            print("BIENESTAR INSTITUCIONAL: conexión fallida, no se suscribe:", reason_code)
            return
        print("BIENESTAR INSTITUCIONAL conectado al broker MQTT")
        client.subscribe(TOPICS["detector"], qos=1)
        print("Suscrito a alertas del Detector.")

    def al_recibir(client, userdata, msg):
        try:
            evento = json.loads(msg.payload.decode("utf-8"))
        except Exception as exc:
            print("EVENTO_RECHAZADO en Bienestar: JSON no parseable:", exc)
            return

        datos = evento.get("data") or {}
        print()
        print("================================")
        print("     BIENESTAR INSTITUCIONAL")
        print("================================")
        print("Alerta recibida:", evento.get("event_name"))
        print("Event ID:", evento.get("event_id"))
        print("Estudiante:", datos.get("codigo_estudiante"), "-", datos.get("nombre"))
        print("Prioridad:", datos.get("nivel_prioridad"))
        print("Señales:", datos.get("senales"))
        print("Acción sugerida:", datos.get("accion_sugerida"))

        acuse = {
            "action": "CasoRegistradoBienestar",
            "event_id_origen": evento.get("event_id"),
            "codigo_estudiante": datos.get("codigo_estudiante"),
            "estado_caso": (
                "CERRADO_POR_ALERTA"
                if evento.get("event_name") == "AlertaCerrada"
                else "PENDIENTE_REVISION"
            ),
            "timestamp_procesamiento": datetime.now().isoformat(timespec="seconds"),
        }
        print()
        print("JSON de acuse de Bienestar:")
        publicar(TOPICS["bienestar"], acuse)
        print("================================")
        print()

    cliente = mqtt.Client(
        mqtt.CallbackAPIVersion.VERSION2,
        client_id=f"bienestar_{uuid.uuid4().hex[:8]}",
    )
    cliente.on_connect = al_conectar
    cliente.on_message = al_recibir
    cliente.connect(BROKER, PORT, 60)
    cliente.loop_start()
    CLIENTES["bienestar"] = cliente
    time.sleep(0.8)
    print("================================")
    print("   BIENESTAR INSTITUCIONAL")
    print("================================")
    print("Escuchando con loop_start().")
    print("Nota: en un kernel aparte se usaría loop_forever(), como en el laboratorio.")
    return cliente


iniciar_bienestar()

## Productor: Sistema Académico

Para la **demo de clase** ejecutar Detector, Bienestar y luego el escenario automático (sección 15). Esta celda es el análogo del laboratorio: usa `input()`.

Si el primer campo queda vacío (Enter), **no publica** por teclado y no pide más datos. El escenario y las pruebas llaman `publicar_rendimiento()`, `publicar_cancelaciones()` y `publicar_recuperado()` sin `input()`.

In [ ]:
print("==== SISTEMA ACADÉMICO ====")
print("Demo de clase: Detector + Bienestar + escenario de la sección 15.")
print("Enter vacío en el primer campo: no se publica desde esta celda.")
print()

codigo = input("Código de estudiante: ")
if not codigo.strip():
    print("Entrada vacía. Use publicar_rendimiento(...) o el escenario automático.")
else:
    nombre = input("Nombre: ") or "Ana Pérez"
    programa = input("Programa: ") or "Ingeniería de Sistemas"
    anterior = input("Promedio anterior: ") or "4.2"
    actual = input("Promedio actual: ") or "2.9"
    publicar_rendimiento(
        codigo.strip(),
        nombre.strip(),
        float(anterior),
        float(actual),
        programa=programa.strip(),
    )
    extra = input("¿Publicar también cancelaciones? (s/N): ")
    if extra.strip().lower() == "s":
        raw = input("Asignaturas canceladas (separadas por coma): ") or "Cálculo I, Física"
        lista = [item.strip() for item in raw.split(",") if item.strip()]
        total = input("Total canceladas: ") or str(len(lista))
        ct = input("¿Cancelación total del periodo? (s/N): ")
        publicar_cancelaciones(
            codigo.strip(),
            nombre.strip(),
            lista,
            int(total),
            cancelacion_total=ct.strip().lower() == "s",
        )

## Productor: Plataforma Virtual

Misma regla que Académico. Para la demo de clase no hace falta esta celda: el escenario llama `publicar_inactividad()`. Enter vacío en el primer campo omite la publicación interactiva.

In [ ]:
print("==== PLATAFORMA VIRTUAL ====")
print("Demo de clase: Detector + Bienestar + escenario de la sección 15.")
print("Enter vacío en el primer campo: no se publica desde esta celda.")
print()

codigo = input("Código de estudiante: ")
if not codigo.strip():
    print("Entrada vacía. Use publicar_inactividad(...) o el escenario automático.")
else:
    nombre = input("Nombre: ") or "Ana Pérez"
    dias = input("Días de inactividad: ") or "18"
    participacion = input("Participación (%): ") or "22"
    publicar_inactividad(
        codigo.strip(),
        nombre.strip(),
        int(dias),
        float(participacion),
    )

## Productor: Sistema Financiero

Misma regla. El paso opcional de mora del escenario usa `publicar_mora()`. Enter vacío omite la publicación interactiva.

In [ ]:
print("==== SISTEMA FINANCIERO ====")
print("Demo de clase: Detector + Bienestar + escenario de la sección 15.")
print("Enter vacío en el primer campo: no se publica desde esta celda.")
print()

codigo = input("Código de estudiante: ")
if not codigo.strip():
    print("Entrada vacía. Use publicar_mora(...) o el escenario automático.")
else:
    nombre = input("Nombre: ") or "Ana Pérez"
    hecho = input("Hecho (mora / pago): ") or "mora"
    if hecho.strip().lower().startswith("p"):
        publicar_pago_regularizado(codigo.strip(), nombre.strip())
    else:
        dias = input("Días de mora: ") or "20"
        valor = input("Valor pendiente: ") or "850000"
        publicar_mora(
            codigo.strip(),
            nombre.strip(),
            int(dias),
            float(valor),
        )

# 15. Escenario mínimo de demostración

Estudiante ficticia **EST001 Ana Pérez**, periodo **2026-2**. Recorre origen → canal → correlación → Bienestar.

| Paso | Qué ocurre | Resultado esperado |
|---|---|---|
| 1 | Arrancar Detector y Bienestar (celdas previas) | ambos escuchan con `loop_start()` |
| 2 | Académico: promedio 4.2 → 2.9 (`delta -1.3`) | 1 señal `CAIDA_RENDIMIENTO`; **sin alerta** |
| 3 | Virtual: 18 días de inactividad | 2 señales; **alerta MEDIA** `EstudianteRequiereRevision`; Bienestar imprime el caso |
| 4 | Opcional. Financiero: 20 días de mora | tercera señal; **AlertaActualizada ALTA** |

Un solo evento no alerta. Eso demuestra correlación (REQ-10), no un condicional local en Académico.

In [ ]:
print("==== ESCENARIO MÍNIMO EST001 ====")
reset_estado()
time.sleep(1)

print()
print("--- Paso 2. Académico: caída 4.2 → 2.9 ---")
publicar_rendimiento("EST001", "Ana Pérez", 4.2, 2.9)
time.sleep(1)

print()
print("--- Paso 3. Virtual: 18 días de inactividad ---")
publicar_inactividad("EST001", "Ana Pérez", 18, 22)
time.sleep(1)

print()
print("--- Paso 4 (opcional). Financiero: 20 días de mora ---")
publicar_mora("EST001", "Ana Pérez", 20, 850000)
time.sleep(1)

print()
print("Estado en memoria de EST001:")
print(ESTADO.get("EST001"))
print("==== FIN DEL ESCENARIO ====")

# 16. Pruebas

Metodología. Cada caso reinicia `ESTADO`. Detector y Bienestar deben estar en ejecución, salvo el caso de fallo, que detiene Bienestar a propósito. Tras ejecutar, el tester completa **Obtenido** y **Evidencia** en la tabla final.

| Caso | Qué se quiere ver |
|---|---|
| Normal | dos hechos válidos producen alerta y acuse de Bienestar |
| Datos incorrectos | rechazo sin tumbar el Detector |
| Fallo | Bienestar detenido; el Detector y los productores siguen |
| Actualización | `RendimientoRecuperado` retira la señal y cierra o actualiza la alerta |

In [ ]:
print("==== PRUEBA NORMAL EST002 ====")
reset_estado()
time.sleep(0.5)
publicar_rendimiento("EST002", "Carlos Ruiz", 4.0, 2.7)
time.sleep(1)
publicar_inactividad("EST002", "Carlos Ruiz", 16, 20)
time.sleep(1)
print("Estado EST002:", ESTADO.get("EST002"))
print("Esperado: alerta MEDIA con CAIDA_RENDIMIENTO e INACTIVIDAD_VIRTUAL, y acuse de Bienestar.")

In [ ]:
print("==== PRUEBA DATOS INCORRECTOS ====")
reset_estado()
time.sleep(0.5)

print()
print("--- Payload sin codigo_estudiante ---")
evento_sin_codigo = envoltorio(
    "MoraFinancieraDetectada",
    "financiero",
    PERIODO,
    {"nombre": "Sin código", "dias_mora": 20, "valor_pendiente": 1000, "estado_obligacion": "EN_MORA"},
)
publicar(TOPICS["financiero"], evento_sin_codigo)
time.sleep(1)

print()
print("--- dias_mora no numérico ---")
evento_abc = envoltorio(
    "MoraFinancieraDetectada",
    "financiero",
    PERIODO,
    {
        "codigo_estudiante": "EST099",
        "nombre": "Prueba Inválida",
        "dias_mora": "abc",
        "valor_pendiente": 1000,
        "estado_obligacion": "EN_MORA",
    },
)
publicar(TOPICS["financiero"], evento_abc)
time.sleep(1)

print()
print("--- JSON no parseable ---")
publicar_crudo(TOPICS["academico"], "{esto no es json")
time.sleep(1)

print("Esperado: tres EVENTO_RECHAZADO. Detector vivo. ESTADO sin alertas nuevas.")
print("ESTADO:", dict(ESTADO))

In [ ]:
print("==== PRUEBA DE FALLO: se detiene Bienestar ====")
reset_estado()
detener_cliente("bienestar")
time.sleep(0.5)

print()
print("Bienestar está detenido. Se publica un flujo que sí debe alertar.")
publicar_rendimiento("EST004", "Luisa Mora", 3.9, 2.5)
time.sleep(1)
publicar_inactividad("EST004", "Luisa Mora", 15, 10)
time.sleep(1)

print()
print("Esperado: el Detector publica EstudianteRequiereRevision. Académico y Virtual no fallan.")
print("Bienestar no imprime acuse mientras está detenido.")
print("Estado EST004:", ESTADO.get("EST004"))
print()
print("Limitación: MQTT en este prototipo no usa persistencia ni mensajes retained.")
print("Si Bienestar no estaba suscrito, no recupera la alerta al reconectar.")

print()
print("Se reinicia Bienestar para las pruebas siguientes. El mensaje ya publicado no se reenvía.")
iniciar_bienestar()

In [ ]:
print("==== PRUEBA DE ACTUALIZACIÓN EST005 ====")
reset_estado()
time.sleep(0.5)
publicar_rendimiento("EST005", "Diego León", 4.1, 2.8)
time.sleep(1)
publicar_inactividad("EST005", "Diego León", 18, 25)
time.sleep(1)
print("Tras dos señales, debe existir alerta activa:", ESTADO.get("EST005"))
print()
print("--- RendimientoRecuperado retira CAIDA_RENDIMIENTO ---")
publicar_recuperado("EST005", "Diego León", 3.8)
time.sleep(1)
print("Estado EST005:", ESTADO.get("EST005"))
print("Esperado: queda INACTIVIDAD_VIRTUAL (18 días, no crítica). AlertaCerrada.")

## Informe de pruebas (etapa 16)

Completado por el tester el 2026-09-20 contra `broker.emqx.io:1883` (celdas 18+19+21+23+31+33–36, sin `input()`). JSON del `.ipynb` válido. Re-ejecución tras el parche de `AlertaActualizada` (no republicar el mismo set de señales).

| Caso | Entrada | Acción | Resultado esperado | Resultado obtenido | Evidencia |
|---|---|---|---|---|---|
| Normal | EST002, promedio 4.0→2.7 e inactividad 16 días | dos `publish` con Detector y Bienestar activos | `EstudianteRequiereRevision` MEDIA con esas dos señales y acuse `CasoRegistradoBienestar` | PASS. Tras 1 señal no alerta; con 2 señales alerta MEDIA `CAIDA_RENDIMIENTO`+`INACTIVIDAD_VIRTUAL`; Bienestar acusa `CasoRegistradoBienestar` / `PENDIENTE_REVISION` | 15:53:44–15:53:47. `event_id` alerta `5dc5445b-98fe-4533-82e3-b5c97cc407a8`. ESTADO EST002 `alerta_activa=True` |
| Datos incorrectos | mora sin `codigo_estudiante`; `dias_mora: "abc"`; payload no JSON | tres publicaciones inválidas | `EVENTO_RECHAZADO` en cada una; Detector sigue vivo; sin alerta nueva | PASS. Tres rechazos; Detector siguió procesando; `ESTADO: {}` | `data.codigo_estudiante vacío o ausente`; `dias_mora no es numérico`; `JSON no parseable: Expecting property name enclosed in double quotes`. Sin `EstudianteRequiereRevision` en este caso |
| Fallo | EST004 con dos señales válidas | detener Bienestar y publicar | Detector publica la alerta; productores no fallan; Bienestar no acusa; al reconectar no hay persistencia | PASS. Detector publicó `EstudianteRequiereRevision` MEDIA; Académico/Virtual publicaron OK; ningún acuse mientras Bienestar estaba detenido; al `iniciar_bienestar()` no reapareció la alerta | 15:54:04–15:54:07. `Cliente 'bienestar' detenido`. Alerta `e1f8f9f3-a6bd-4a79-b954-aa8ace2e8b27`. Tras reconectar solo el banner de escucha, sin acuse de EST004 |
| Actualización | EST005 alertado y luego `RendimientoRecuperado` | publicar recuperación de promedio 3.8 | se retira `CAIDA_RENDIMIENTO`; queda una señal no crítica; `AlertaCerrada` | PASS. Tras 2 señales alerta activa; `RendimientoRecuperado` deja solo `INACTIVIDAD_VIRTUAL` (18 días, no crítica) y publica `AlertaCerrada`; Bienestar `CERRADO_POR_ALERTA` | 15:54:15–15:54:19. Alerta inicial `ecfadff7-bca0-44d5-9c68-101833517351`; cierre `8c47b3be-ca18-4dbe-bb08-e22e2301b712` |

Escenario mínimo EST001 (celda 31), mismo runner: 1 señal no alerta; 2 señales `EstudianteRequiereRevision` MEDIA (`b2ae9106-3fbd-42e6-bb71-0759562c48a7`); mora 20 días `AlertaActualizada` ALTA (`efccf73e-90d0-4255-92dc-68617fb142f9`) porque sí añade `MORA_FINANCIERA`. PASS.

Extra (harness, no producto): con alerta activa, republicar el mismo rendimiento y la misma inactividad no dispara `AlertaActualizada` (`Señales y prioridad sin cambio. No se republica AlertaActualizada.` dos veces). En EST002/EST004/EST005 el escenario no republicó el mismo set; la única `AlertaActualizada` de la corrida fue la mora de EST001.


# Limitaciones del prototipo y evolución

El recorte es suficiente para demostrar interoperabilidad. No es la plataforma institucional.

| Limitación actual | Evolución posible |
|---|---|
| Broker público EMQX, sin SLA de la universidad | broker institucional administrado, con redes internas |
| Sin persistencia ni *retained*; si Bienestar no está, el aviso se pierde | cola durable, *retained* puntual o bandeja de alertas |
| Identidad = código ficticio en el mensaje | identidad federada (código institucional único, sin cédula en el bus) |
| QoS 1 sin política de reintento de negocio | QoS acordado, idempotencia por `event_id`, *dead letter* |
| Sin autenticación ni cifrado de aplicación | TLS, cuentas por sistema, autorización por topic |
| `ESTADO` solo en memoria de un proceso | historial de casos, auditoría y reconstrucción tras reinicio |
| Un periodo y umbrales fijos de demo | umbrales por programa y calendario académico |
| Stubs en un cuaderno | adaptadores junto a cada sistema real, sin GUI en el bus |

Estas limitaciones no anulan REQ-01 a REQ-11 en el recorte de curso. Sí impiden pretender que el prototipo es el bus de producción.

# 18. Preguntas de sustentación

Respuestas listas para leer en voz alta. Cada una se ata al caso de la estudiante, no a teoría genérica.

### 1. ¿Cuál era el problema?

Bienestar Institucional identifica tarde a quienes necesitan acompañamiento. La información ya existe: promedio y cancelaciones en Académico, inactividad en la plataforma virtual, mora en Financiero. Esos sistemas no se hablan. En el ejemplo, Ana Pérez puede bajar de 4.2 a 2.9, ausentarse dieciocho días y entrar en mora, y Bienestar se entera solo si ella pide ayuda o si un docente remite. El problema no es falta de software de negocio. Es falta de interoperabilidad oportuna.

### 2. ¿Qué información estaba involucrada?

Se intercambia el mínimo para decidir una revisión. De Académico: promedios, delta y cancelaciones. De Virtual: días de inactividad y participación. De Financiero: días de mora y regularización. Del Detector: la alerta con prioridad, señales y `event_id` de origen. Bienestar responde con el acuse del caso. No viajan cédulas ni expedientes. El cruce es el código ficticio del estudiante.

### 3. ¿Qué sistemas necesitaban interoperar?

Los cuatro del enunciado. Académico, Virtual y Financiero producen los hechos. Bienestar es el destino institucional. El Detector es un componente de integración, no un quinto sistema de atención. Se usaron los cuatro porque omitir uno dejaría la regla de alerta a ciegas: el PDF describe precisamente la confluencia de esas fuentes.

### 4. ¿Qué alternativas fueron consideradas?

La A: hechos asíncronos y un correlador. Cada silo emite cuando ocurre el hecho; el Detector decide; Bienestar solo escucha la alerta. La B: Bienestar consulta por APIs síncronas a los tres núcleos. La B da un expediente al preguntar, pero acopla tiempos y contratos, y no avisa cuando Ana deja de entrar al aula. Se compararon oportunidad, acoplamiento, fallo e incorporación de un sistema nuevo.

### 5. ¿Por qué seleccionaron su arquitectura?

Por coherencia con el problema y con los requisitos, no por moda. El enunciado dice que los hechos no ocurren al mismo tiempo y que no se reemplazan sistemas. REQ-01, REQ-02, REQ-03, REQ-06, REQ-07 y REQ-10 piden aviso al ocurrir, núcleos intactos, tolerancia al fallo de un consumidor y correlación. Eso es la alternativa A. Un `if` solo en Académico no sería interoperabilidad.

### 6. ¿Por qué seleccionaron las tecnologías utilizadas?

Después de fijar la arquitectura. Hacía falta un canal publicar/suscribir, un contrato legible y cero servidores que el equipo no controla. Se comparó MQTT, REST y Kafka; `paho-mqtt` frente a otros clientes; JSON frente a XML; EMQX público frente a Mosquitto local. Ganó EMQX + paho + JSON + Python por fidelidad a A, esfuerzo de prototipo y visibilidad del mensaje. No es el MQTT institucional.

### 7. ¿Cómo funciona la solución?

Un hecho ocurre en el origen y se publica en su topic. El Detector valida el JSON y el código de estudiante. Actualiza `ESTADO` y evalúa señales. Una sola caída de nota no alerta. Con dos señales, o con una crítica, publica `EstudianteRequiereRevision`. Bienestar imprime el caso y un JSON `CasoRegistradoBienestar`. Si llega mora después, actualiza. Si el promedio se recupera y ya no hay regla, cierra.

### 8. ¿Qué ocurre cuando se presenta un fallo?

Si el mensaje es inválido, se imprime `EVENTO_RECHAZADO` y el Detector sigue. Si Bienestar se detiene, Académico, Virtual y el Detector continúan; la alerta se publica igual. Lo que no hace este prototipo es guardar el aviso: MQTT aquí va sin persistencia ni *retained*. Al reconectar, Bienestar no recupera lo perdido. Eso se muestra en la prueba de fallo y se deja como evolución.

### 9. ¿Qué limitaciones tiene el prototipo?

Estado en memoria, broker público, sin autenticación, identificadores ficticios, umbrales fijos, un periodo y sin interfaz. Depende de Internet: si EMQX no responde, la demo no corre. No sustituye a los sistemas reales. Es un recorte para ver el recorrido del hecho hasta Bienestar, que es lo que pide la sustentación.

### 10. ¿Cómo podría evolucionar la solución?

Pasar el canal a un broker institucional con TLS y cuentas por sistema. Persistir alertas y armar historial de casos. Federar la identidad sin poner cédulas en el bus. Ajustar QoS, idempotencia por `event_id` y políticas de retención. Calibrar umbrales por programa. Colocar adaptadores junto a cada núcleo, sin GUI en la integración. El Detector seguiría siendo correlador, no un sistema de atención.

# Cómo detener los clientes

Los consumidores quedan en hilos con `loop_start()`. Antes de cerrar el cuaderno, o antes de volver a ejecutar el Detector y Bienestar desde cero, detener:

```python
detener_cliente("detector")
detener_cliente("bienestar")
# equivalente:
# cliente.loop_stop()
# cliente.disconnect()
```

`detener_todos()` corta todos los clientes guardados en `CLIENTES`. Volver a las celdas de Detector y Bienestar los arranca de nuevo, cada uno con un `client_id` corto de `uuid`, para no chocar en el broker público.